# 02 Wikipedia Evidence Retrieval


In [ ]:
import os
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer


## 1. Paths and configuration

In [ ]:
FAISS_PATH = Path("../WIKI_resource/wiki_retrieval_output/wiki_passages.faiss")
PASSAGES_PATH = Path("../WIKI_resource/wiki_retrieval_output/wiki_passages.csv")

CLAIMS_PATH = Path("../data/Phi3_Claims_Level_Wikipedia_Only.csv")

OUTPUT_PATH = Path("../data/Phi3_Claims_FAISS_Evidence.csv")

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("FAISS:", FAISS_PATH)
print("Passages:", PASSAGES_PATH)
print("Claims:", CLAIMS_PATH)
print("Output:", OUTPUT_PATH)


## 2. Load saved FAISS index and Wikipedia passages

In [ ]:
index = faiss.read_index(str(FAISS_PATH))
passages_df = pd.read_csv(PASSAGES_PATH)

print("FAISS vectors:", index.ntotal)
print("FAISS dimension:", index.d)
print("Passages:", len(passages_df))

assert index.ntotal == len(passages_df), (
    f"Mismatch: FAISS contains {index.ntotal} vectors "
    f"but passages CSV contains {len(passages_df)} rows."
)

passages_df.head()


## 3. Load the same embedding model used to build the index

In [ ]:
model = SentenceTransformer(MODEL_NAME)

embedding_dim = model.get_sentence_embedding_dimension()

print("Embedding model:", MODEL_NAME)
print("Embedding dimension:", embedding_dim)

assert index.d == embedding_dim, (
    f"Dimension mismatch: FAISS={index.d}, model={embedding_dim}"
)


## 4. Define FAISS retrieval function

In [ ]:
def search_claim(claim, k=TOP_K):
    claim = str(claim).strip()

    query_embedding = model.encode(
        [claim],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    valid = indices[0] >= 0
    result_indices = indices[0][valid]
    result_scores = scores[0][valid]

    results = passages_df.iloc[result_indices].copy().reset_index(drop=True)
    results["score"] = result_scores

    return results


## 5. Quick test

In [ ]:
test_claim = "Vampires are mythical creatures."

test_results = search_claim(test_claim, k=TOP_K)

display_cols = [
    c for c in ["page_title", "passage_text", "source_url", "score"]
    if c in test_results.columns
]

test_results[display_cols]


## 6. Load atomic claims

In [ ]:
claims_df = pd.read_csv(CLAIMS_PATH)

print("Rows:", len(claims_df))
print("Columns:", claims_df.columns.tolist())

if "Atomic_Claim" not in claims_df.columns:
    raise ValueError("Input CSV must contain an 'Atomic_Claim' column.")

empty_mask = (
    claims_df["Atomic_Claim"].isna()
    | claims_df["Atomic_Claim"].fillna("").astype(str).str.strip().eq("")
)

print("Empty claims:", int(empty_mask.sum()))
print("Non-empty claims:", int((~empty_mask).sum()))

claims_df.head()


## 7. Retrieve Top-K evidence for every claim


In [ ]:
retrieval_rows = []

for _, row in tqdm(claims_df.iterrows(), total=len(claims_df)):

    output = row.to_dict()
    raw_claim = row["Atomic_Claim"]

    # Claim extraction failure
    if pd.isna(raw_claim) or not str(raw_claim).strip():
        output["Retrieval_Status"] = "SKIPPED_EMPTY_CLAIM"

        for rank in range(1, TOP_K + 1):
            output[f"Evidence_{rank}"] = ""
            output[f"Evidence_{rank}_Score"] = np.nan
            output[f"Evidence_{rank}_Page"] = ""
            output[f"Evidence_{rank}_URL"] = ""

        retrieval_rows.append(output)
        continue

    claim = str(raw_claim).strip()

    try:
        results = search_claim(claim, k=TOP_K)

        output["Retrieval_Status"] = "SUCCESS"

        for rank in range(1, TOP_K + 1):
            if rank <= len(results):
                ev = results.iloc[rank - 1]

                output[f"Evidence_{rank}"] = ev.get("passage_text", "")
                output[f"Evidence_{rank}_Score"] = ev.get("score", np.nan)
                output[f"Evidence_{rank}_Page"] = ev.get("page_title", "")
                output[f"Evidence_{rank}_URL"] = ev.get("source_url", "")
            else:
                output[f"Evidence_{rank}"] = ""
                output[f"Evidence_{rank}_Score"] = np.nan
                output[f"Evidence_{rank}_Page"] = ""
                output[f"Evidence_{rank}_URL"] = ""

    except Exception as e:
        output["Retrieval_Status"] = "ERROR"
        output["Retrieval_Error"] = str(e)

        for rank in range(1, TOP_K + 1):
            output[f"Evidence_{rank}"] = ""
            output[f"Evidence_{rank}_Score"] = np.nan
            output[f"Evidence_{rank}_Page"] = ""
            output[f"Evidence_{rank}_URL"] = ""

    retrieval_rows.append(output)

retrieval_df = pd.DataFrame(retrieval_rows)

print("\nRetrieval status:")
print(retrieval_df["Retrieval_Status"].value_counts(dropna=False))


## 8. Inspect retrieval results

In [ ]:
display_cols = [
    c for c in [
        "Question_ID",
        "Claim_ID",
        "Atomic_Claim",
        "Retrieval_Status",
        "Evidence_1_Page",
        "Evidence_1_Score",
        "Evidence_1",
        "Evidence_2_Page",
        "Evidence_2_Score",
        "Evidence_3_Page",
        "Evidence_3_Score",
    ]
    if c in retrieval_df.columns
]

retrieval_df[display_cols].head(20)


## 9. Save evidence

In [ ]:
retrieval_df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(retrieval_df))


## 10. Retrieval summary

In [ ]:
print("Total rows:", len(retrieval_df))

print("\nStatus:")
print(retrieval_df["Retrieval_Status"].value_counts(dropna=False))

for rank in range(1, TOP_K + 1):
    score_col = f"Evidence_{rank}_Score"

    if score_col in retrieval_df.columns:
        scores = pd.to_numeric(retrieval_df[score_col], errors="coerce")
        print(
            f"\nTop-{rank} score:"
            f" mean={scores.mean():.4f},"
            f" median={scores.median():.4f}"
        )
